In [1]:
import struct
from pathlib import Path
from collections import defaultdict

In [3]:

CLASSES = [
    "Ball out of play", "Clearance", "Corner", "Direct free-kick",
    "Foul", "Goal", "Indirect free-kick", "Kick-off", "Offside",
    "Shots off target", "Shots on target", "Substitution",
    "Throw-in", "Yellow card"
]
CLASS_TO_ID = {cls: i for i, cls in enumerate(CLASSES)}


def get_tfrecord_counts(tfrecord_paths):
    """
    Count per-class records across multiple TFRecord files/directories.
    
    Args:
        tfrecord_paths: list of Path objects — can be individual .tfrecord 
                        files or directories containing .tfrecord files
    """
    from tensorflow.core.example.example_pb2 import Example

    counts = defaultdict(int)
    files_read = []

    # Collect all .tfrecord files from the provided paths
    for path in tfrecord_paths:
        path = Path(path)
        if path.is_dir():
            found = list(path.glob("*.tfrecord"))
            files_read.extend(found)
        elif path.is_file() and path.suffix == ".tfrecord":
            files_read.append(path)
        else:
            print(f"Path not found or not a TFRecord: {path}")

    if not files_read:
        print("No existing TFRecord files found, starting fresh.")
        return counts

    print(f"Reading counts from {len(files_read)} TFRecord file(s):")
    for f in files_read:
        print(f"  {f}")

    for tfrecord_file in files_read:
        if tfrecord_file.stat().st_size == 0:
            print(f"Skipping empty file: {tfrecord_file}")
            continue

        file_counts = defaultdict(int)
        try:
            with open(tfrecord_file, "rb") as f:
                while True:
                    header = f.read(8)
                    if len(header) < 8:
                        break
                    length = struct.unpack("<Q", header)[0]
                    f.read(4)
                    data = f.read(length)
                    f.read(4)
                    if len(data) < length:
                        break
                    example = Example()
                    example.ParseFromString(data)
                    label_id = example.features.feature["label"].int64_list.value[0]
                    cls = CLASSES[label_id]
                    counts[cls] += 1
                    file_counts[cls] += 1

            total_in_file = sum(file_counts.values())
            print(f"{tfrecord_file.name}: {total_in_file} records")

        except Exception as e:
            print(f" Error reading {tfrecord_file}: {e}, skipping.")

    print("\nCombined counts across all files:")
    for cls in CLASSES:
        print(f"  {cls:<25} {counts[cls]}")
    print(f"  {'TOTAL':<25} {sum(counts.values())}")


In [4]:

get_tfrecord_counts([
    Path(r"D:\Football Event Detection\Dataset\Data\train_merged_v2.tfrecord"),
])


Reading counts from 1 TFRecord file(s):
  D:\Football Event Detection\Dataset\Data\train_merged_v2.tfrecord
 Error reading D:\Football Event Detection\Dataset\Data\train_merged_v2.tfrecord: , skipping.

Combined counts across all files:
  Ball out of play          0
  Clearance                 141
  Corner                    73
  Direct free-kick          39
  Foul                      209
  Goal                      27
  Indirect free-kick        210
  Kick-off                  45
  Offside                   43
  Shots off target          103
  Shots on target           111
  Substitution              53
  Throw-in                  0
  Yellow card               33
  TOTAL                     1087


In [10]:
import tensorflow as tf
from collections import defaultdict

CLASSES = [
    "Ball out of play", "Clearance", "Corner", "Direct free-kick",
    "Foul", "Goal", "Indirect free-kick", "Kick-off", "Offside",
    "Shots off target", "Shots on target", "Substitution",
    "Throw-in", "Yellow card"
]

OUTPUT = r"D:\Football Event Detection\Dataset\Data\train_merged_v2.tfrecord"
counts = defaultdict(int)

feature_spec = {
    "label": tf.io.FixedLenFeature([], tf.int64),
    "video": tf.io.FixedLenFeature([], tf.string),  # parse but ignore
}

dataset = tf.data.TFRecordDataset(OUTPUT, buffer_size=256 * 1024 * 1024)

for i, raw in enumerate(dataset):
    parsed = tf.io.parse_single_example(raw, {"label": tf.io.FixedLenFeature([], tf.int64)})
    label_id = parsed["label"].numpy()
    counts[CLASSES[label_id]] += 1
    if (i + 1) % 1000 == 0:
        print(f"  Processed {i+1} records...", end="\r")

print(f"\nFinal class counts:")
for cls in CLASSES:
    print(f"  {cls:<25} {counts[cls]}")
print(f"  {'TOTAL':<25} {sum(counts.values())}")

DataLossError: {{function_node __wrapped__IteratorGetNext_output_types_1_device_/job:localhost/replica:0/task:0/device:CPU:0}} corrupted record at 2615673366 [Op:IteratorGetNext]

In [6]:
import struct
from pathlib import Path
from collections import defaultdict
from tensorflow.core.example.example_pb2 import Example

CLASSES = [
    "Ball out of play", "Clearance", "Corner", "Direct free-kick",
    "Foul", "Goal", "Indirect free-kick", "Kick-off", "Offside",
    "Shots off target", "Shots on target", "Substitution",
    "Throw-in", "Yellow card"
]

OUTPUT = Path(r"D:\Football Event Detection\Dataset\Data\train_merged_v2.tfrecord")
counts = defaultdict(int)
total = 0

CHUNK = 16 * 1024 * 1024  # read 16MB at a time to avoid MemoryError

with open(OUTPUT, "rb") as f:
    while True:
        header = f.read(8)
        if len(header) < 8:
            break

        length = struct.unpack("<Q", header)[0]
        f.read(4)  # length crc

        # Read record in chunks to avoid loading full ~7MB at once
        data = bytearray()
        remaining = length
        while remaining > 0:
            chunk = f.read(min(CHUNK, remaining))
            if not chunk:
                break
            data.extend(chunk)
            remaining -= len(chunk)

        f.read(4)  # data crc

        if len(data) < length:
            break

        try:
            example = Example()
            example.ParseFromString(bytes(data))
            label_id = example.features.feature["label"].int64_list.value[0]
            counts[CLASSES[label_id]] += 1
            total += 1
        except Exception as e:
            print(f"Parse error at record {total}: {e}")
            continue

        if total % 500 == 0:
            print(f"  Processed {total} records...", end="\r")

print(f"\nFinal class counts:")
for cls in CLASSES:
    print(f"  {cls:<25} {counts[cls]}")
print(f"  {'TOTAL':<25} {sum(counts.values())}")

  Processed 26500 records...
Final class counts:
  Ball out of play          2008
  Clearance                 2012
  Corner                    2023
  Direct free-kick          1584
  Foul                      2013
  Goal                      1557
  Indirect free-kick        2010
  Kick-off                  2050
  Offside                   1735
  Shots off target          1991
  Shots on target           1999
  Substitution              2107
  Throw-in                  1994
  Yellow card               1737
  TOTAL                     26820


In [3]:
import os
path = r"D:\Football Event Detection\Dataset\Data\train_merged_v2.tfrecord"
print(f"File size: {os.path.getsize(path) / (2**30):.2f} GB")

File size: 46.89 GB


In [4]:
from pathlib import Path
p = Path(r"D:\Football Event Detection\Dataset\Data\train_merged_v2.tfrecord")
if p.exists():
    p.unlink()
    print("Deleted.")
else:
    print("Does not exist — safe to merge.")

Deleted.


In [9]:
import tensorflow as tf
count = sum(1 for _ in tf.data.TFRecordDataset(str(r"D:\Football Highlight Generation\TFRecords\val.tfrecord")))
print(f"Val records: {count}")

Val records: 1496
